In [1]:
# ===============================================
# CELL 1: SETUP & INSTALL PACKAGES — FIX ALL
# ===============================================

# Gỡ hết các package conflict trước
!pip uninstall -y hdbscan sentence-transformers huggingface_hub gensim scipy bertopic umap-learn -q

# Cài đúng thứ tự, pin version tương thích nhau
!pip install -q scipy==1.11.4
!pip install -q numpy==1.26.4
!pip install -q gensim==4.3.2
!pip install -q huggingface_hub==0.21.0
!pip install -q transformers==4.36.0
!pip install -q sentence-transformers==2.7.0
!pip install -q umap-learn==0.5.5
!pip install -q hdbscan==0.8.33 --no-binary hdbscan
!pip install -q bertopic==0.16.0
!pip install -q tqdm pandas scikit-learn

print("✅ Done! Bây giờ RESTART KERNEL rồi chạy Cell 2")

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires gensim, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires scipy>=1.13, but you have scipy 1.11.4 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.11.4 which is incompatible.
cuml-cu12 26.2.0 requires scipy>=1.13.0, but you have scipy 1.11.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
giddy 2.3.8 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires scipy>=1.13, but you have scipy 1.11.4 which is

In [2]:
import pandas as pd
import os
import sys

# FIX: path đến FOLDER chứa file, không phải path đến file
sys.path.append("/kaggle/input/datasets/hoangquancs04222/data-proccessed-score")
from bertopic_model import VietnameseBERTopicModel

print("📊 LOADING DATA")
print("="*50)

DATA_PATH = "/kaggle/input/datasets/hoangquancs04222/data-proccessed-score/stg_posts_core.csv"

COL_NAMES = [
    "post_id", "source", "type", "author", "parent_id",
    "title", "body", "segmented_text",
    "col9", "col10", "col11", "col12",
    "created_at", "crawled_at", "col15", "url"
]

df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=COL_NAMES,
    sep=",",
    on_bad_lines="skip",
    engine="python"
)

print(f"✅ Loaded: {DATA_PATH}")
print(f"   Shape: {df.shape}")

documents = (
    df["segmented_text"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

post_ids = df.loc[df["segmented_text"].notna(), "post_id"].astype(str).tolist()

valid_mask = [len(d) > 10 for d in documents]
documents = [d for d, ok in zip(documents, valid_mask) if ok]
post_ids  = [p for p, ok in zip(post_ids, valid_mask) if ok]

print(f"\n📈 DATA STATS:")
print(f"   Total documents: {len(documents):,}")
print(f"   Avg length: {sum(len(d) for d in documents)/len(documents):.0f} chars")
print(f"\n📝 SAMPLE:")
for i, doc in enumerate(documents[:3]):
    print(f"   {i+1}. {doc[:120]}...")

2026-05-01 11:47:09.197714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777636029.221220     427 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777636029.229549     427 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777636029.249165     427 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777636029.249182     427 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777636029.249184     427 computation_placer.cc:177] computation placer alr

📊 LOADING DATA
✅ Loaded: /kaggle/input/datasets/hoangquancs04222/data-proccessed-score/stg_posts_core.csv
   Shape: (172200, 16)

📈 DATA STATS:
   Total documents: 171,480
   Avg length: 321 chars

📝 SAMPLE:
   1. microsoft vừa công_bố một bản cập_nhật quan_trọng sắp tới cho windows 11 trong đó sẽ thay_thế ô tìm_kiếm windows search ...
   2. chỉ còn một tuần nữa là đến ngày ra_mắt chính_thức nhưng toàn_bộ thông_số kỹ_thuật chi_tiết của oneplus ace 6 dường_như ...
   3. hôm_qua 20 10 iqoo đã chính_thức ra_mắt mẫu iqoo 15 tại thị_trường trung_quốc đây là một bản nâng_cấp toàn_diện so với t...


In [3]:
# ===============================================
# CELL 3: COMPUTE PHOBERT EMBEDDINGS (CACHE)
# ===============================================

import numpy as np, random, time, torch
from sklearn.preprocessing import normalize
print("🧪 COMPUTING PHOBERT EMBEDDINGS")
print("="*50)

random.seed(42)
TUNING_SAMPLE_SIZE = min(5000, len(documents))
tuning_docs = random.sample(documents, TUNING_SAMPLE_SIZE)
tuning_post_ids = random.sample(post_ids, TUNING_SAMPLE_SIZE)  # giữ khớp thứ tự

# Dùng VietnameseBERTopicModel.encode() — nhất quán với production
tmp_model = VietnameseBERTopicModel(verbose=True)
print(f"\n[sample] {len(tuning_docs):,} docs (random seed=42)")

start = time.time()
embeddings = tmp_model.encode(tuning_docs, batch_size=32)

embeddings = normalize(embeddings)
print(f"\n✅ Embeddings: {embeddings.shape}, dtype: {embeddings.dtype}")
print(f"   Time: {(time.time()-start)/60:.1f} min")
print("✅ Embeddings đã được chuẩn hóa L2!")
# Giải phóng VRAM
del tmp_model
torch.cuda.empty_cache()

🧪 COMPUTING PHOBERT EMBEDDINGS
🚀 GPU detected: Tesla T4

[1/4] Loading embedding model: vinai/phobert-base


✅ Embedding model loaded (device: cuda)

[2/4] Configuring UMAP
  - n_neighbors: 15
  - n_components: 5
  - min_dist: 0.0
✅ UMAP configured

[3/4] Configuring HDBSCAN
  - min_cluster_size: 15
  - min_samples: 10
✅ HDBSCAN configured

[4/4] Building BERTopic pipeline
✅ BERTopic pipeline ready

Model initialized successfully!


[sample] 5,000 docs (random seed=42)

Encoding 5,000 documents với PhoBERT...
  batch_size=32, device=cuda


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

✅ Embeddings shape: (5000, 768), dtype: float32

✅ Embeddings: (5000, 768), dtype: float32
   Time: 0.4 min
✅ Embeddings đã được chuẩn hóa L2!


In [4]:
# ===============================================
# CELL 4-5-6: EXPERIMENTS (giữ nguyên từ notebook)
# ===============================================

# Chỉ cần thêm import ở đầu nếu chưa có
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# ---- Paste nguyên Cell 4 (experiments list) từ notebook vào đây ----
# ===============================================
# CELL 4: DEFINE EXPERIMENT GRID (UPDATED)
# ===============================================

print("🔬 EXPERIMENT GRID - GRANULAR TOPICS FOCUS")
print("="*50)

# Define experiments based on new strategy
experiments = [
    # 0. Baseline (để so sánh)
    {
        'name': 'baseline',
        'n_neighbors': 15,
        'min_dist': 0.0,
        'min_cluster_size': 15,
        'min_samples': 10,
        'nr_topics': 'auto',
        'description': 'Task 3.1 default settings'
    },
    # 1. Golden Config (Được đề xuất)
    {
        'name': 'exp_1_golden_config',
        'n_neighbors': 30,
        'min_dist': 0.01,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 'auto',
        'description': 'Golden: Balanced granular topics, very few outliers'
    },
    # 2. Ultra Granular (Bắt các topic siêu nhỏ)
    {
        'name': 'exp_2_ultra_granular',
        'n_neighbors': 30,
        'min_dist': 0.0,
        'min_cluster_size': 5,
        'min_samples': 1,
        'nr_topics': 'auto',
        'description': 'Extreme micro-topics, force 0 outliers'
    },
    # 3. Wider Context (Nhìn tổng quan hơn)
    {
        'name': 'exp_3_wider_context',
        'n_neighbors': 40,
        'min_dist': 0.0,
        'min_cluster_size': 7,
        'min_samples': 2,
        'nr_topics': 'auto',
        'description': 'More neighbors, moderate clusters'
    },
    # 4. Global Granular (Cân bằng nhất)
    {
        'name': 'exp_4_global_granular',
        'n_neighbors': 50,
        'min_dist': 0.0,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 'auto',
        'description': 'Max neighbors for global structure'
    },
    # 5. Spread Golden (Giãn cụm)
    {
        'name': 'exp_5_spread_golden',
        'n_neighbors': 30,
        'min_dist': 0.1,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 'auto',
        'description': 'Golden config but spread clusters (min_dist=0.1)'
    },
    # 6. Force 20 Topics (Dựa trên cấu trúc Golden)
    {
        'name': 'exp_6_force_20',
        'n_neighbors': 30,
        'min_dist': 0.01,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 20,
        'description': 'Force exactly 20 dashboard-ready topics'
    },
    # 7. Force 30 Topics 
    {
        'name': 'exp_7_force_30',
        'n_neighbors': 30,
        'min_dist': 0.01,
        'min_cluster_size': 8,
        'min_samples': 2,
        'nr_topics': 30,
        'description': 'Force exactly 30 detailed topics'
    }
]

# Print summary
print(f"Total experiments: {len(experiments)}")
print("\n" + "-"*85)
print(f"{'Name':<25} {'n_nbr':>6} {'m_dist':>6} {'m_clust':>7} {'m_samp':>6} {'nr_top':>8}")
print("-" * 85)
for exp in experiments:
    print(f"{exp['name']:<25} {exp['n_neighbors']:>6} {exp['min_dist']:>6} {exp['min_cluster_size']:>7} {exp['min_samples']:>6} {str(exp['nr_topics']):>8}")
print("-" * 85)
# ---- Paste nguyên Cell 5 (run experiments loop) vào đây ----
# ===============================================
# CELL 5: RUN EXPERIMENTS
# ===============================================

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import time
import warnings
warnings.filterwarnings('ignore')

print('🏋️ RUNNING EXPERIMENTS')
print('='*50)
print(f'Documents: {len(tuning_docs):,}')
print(f'Embeddings shape: {embeddings.shape}')
print(f'Experiments: {len(experiments)}')
print('='*50)

results = []


def calculate_diversity(topic_model, top_n=10):
    """Calculate topic diversity (unique words ratio)"""
    try:
        topics = topic_model.get_topics()
        all_words = []
        for topic_id, words in topics.items():
            if topic_id != -1:
                all_words.extend([w[0] for w in words[:top_n]])
        if not all_words:
            return 0.0
        return len(set(all_words)) / len(all_words)
    except Exception:
        return 0.0


def calculate_umass(topic_model, texts):
    """
    Tính coherence U_Mass — nhanh hơn C_V (không cần sliding window).
    Dùng trong tuning loop; C_V chỉ tính cho best config cuối cùng.
    U_Mass range: thường âm, gần 0 hơn = tốt hơn.
    """
    try:
        topics_dict = {tid: words for tid, words in topic_model.get_topics().items() if tid != -1}
        if not topics_dict:
            return 0.0
        topics_words = [
            [w for w, _ in words[:10]]
            for words in topics_dict.values()
        ]
        # Tokenize — filter token độ dài <= 1 (nhất quán với bertopic_model.py)
        tokenized = [[t for t in doc.split() if len(t) > 1] for doc in texts]
        tokenized = [t for t in tokenized if t]
        dictionary = Dictionary(tokenized)
        dictionary.filter_extremes(no_below=2, no_above=0.95)
        cm = CoherenceModel(
            topics=topics_words,
            texts=tokenized,
            dictionary=dictionary,
            coherence='u_mass',
        )
        return cm.get_coherence()
    except Exception:
        return 0.0


for i, exp in enumerate(experiments):
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(experiments)}] {exp['name']}")
    print(f"   {exp['description']}")
    print(f"   n_neighbors={exp['n_neighbors']}, min_dist={exp['min_dist']}, "
          f"min_cluster_size={exp['min_cluster_size']}, min_samples={exp['min_samples']}, "
          f"nr_topics={exp['nr_topics']}")

    start_time = time.time()

    try:
        umap_model = UMAP(
            n_neighbors=exp['n_neighbors'],
            n_components=10,
            min_dist=exp['min_dist'],
            metric='cosine',
            random_state=42,
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=exp['min_cluster_size'],
            min_samples=exp['min_samples'],
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
        )

        # FIX: cast nr_topics sang int nếu là số, tránh TypeError trên một số BERTopic versions
        nr_topics_val = None if exp['nr_topics'] == 'auto' else int(exp['nr_topics'])

        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            nr_topics=nr_topics_val,
            calculate_probabilities=False,  # Faster trong tuning
            verbose=False,
        )

        # Dùng pre-computed embeddings — bỏ qua encode PhoBERT
        topics, probs = topic_model.fit_transform(tuning_docs, embeddings=embeddings)

        training_time = time.time() - start_time

        topic_info = topic_model.get_topic_info()
        n_topics = len([t for t in topic_info['Topic'] if t != -1])
        n_outliers = (
            topic_info[topic_info['Topic'] == -1]['Count'].sum()
            if -1 in topic_info['Topic'].values else 0
        )
        outlier_ratio = n_outliers / len(tuning_docs)
        diversity = calculate_diversity(topic_model)
        coherence_umass = calculate_umass(topic_model, tuning_docs)  # FIX: thêm metric

        result = {
            'name': exp['name'],
            'description': exp['description'],
            'n_neighbors': exp['n_neighbors'],
            'min_dist': exp['min_dist'],
            'min_cluster_size': exp['min_cluster_size'],
            'min_samples': exp['min_samples'],
            'nr_topics': str(exp['nr_topics']),
            'n_topics_found': n_topics,
            'n_outliers': n_outliers,
            'outlier_ratio': outlier_ratio,
            'diversity': diversity,
            'coherence_umass': coherence_umass,  # FIX: thêm vào results
            'training_time': training_time,
        }
        results.append(result)

        print(f'   ✅ {training_time:.1f}s | Topics: {n_topics} | '
              f'Outliers: {outlier_ratio*100:.1f}% | '
              f'Diversity: {diversity:.3f} | '
              f'U_Mass: {coherence_umass:.4f}')

        pd.DataFrame(results).to_csv('/kaggle/working/tuning_progress.csv', index=False)

    except Exception as e:
        print(f'   ❌ FAILED: {e}')
        results.append({
            'name': exp['name'],
            'description': exp['description'],
            'error': str(e),
        })

print('\n' + '='*50)
print(f'✅ COMPLETED {len(results)} EXPERIMENTS')
print('='*50)

# ---- Paste nguyên Cell 6 (analyze results) vào đây ----
# ===============================================
# CELL 6: ANALYZE RESULTS
# ===============================================

import pandas as pd

print('📊 EXPERIMENT RESULTS ANALYSIS')
print('='*60)

results_df = pd.DataFrame(results)
valid_results = results_df[results_df['n_topics_found'].notna()].copy()


def composite_score(row):
    """
    Composite score để rank experiments.

    Weights:
        outlier_ratio   → 35% (fewer outliers = better)
        diversity       → 25% (more diverse = better)
        coherence_umass → 25% (U_Mass gần 0 = better; normalize từ âm về [0,1])
        topic_count     → 15% (smooth penalty, peak tại 10 topics)

    FIX so với phiên bản cũ:
        - Thêm coherence_umass thay vì static topic_bonus
        - Smooth penalty topic_count thay vì hard threshold
        - Normalize U_Mass: clip [-20, 0] → [0, 1]
    """
    outlier_score = (1 - row['outlier_ratio']) * 0.35
    diversity_score = row['diversity'] * 0.25

    # U_Mass thường âm, gần 0 hơn = tốt hơn
    # Normalize về [0, 1]: clip [-20, 0] → map tuyến tính
    u_mass_clipped = max(-20.0, min(0.0, row.get('coherence_umass', -20.0)))
    coherence_score = (u_mass_clipped + 20.0) / 20.0 * 0.25

    # Smooth penalty: peak = 1.0 tại 10 topics, giảm dần ra xa
    n = row['n_topics_found']
    topic_score = max(0.0, 1.0 - abs(n - 10) / 20.0) * 0.15

    return outlier_score + diversity_score + coherence_score + topic_score


valid_results['composite_score'] = valid_results.apply(composite_score, axis=1)
valid_results = valid_results.sort_values('composite_score', ascending=False)

print('\n📋 ALL RESULTS (sorted by composite score):')
print('-'*110)
display_cols = ['name', 'n_topics_found', 'outlier_ratio', 'diversity', 'coherence_umass', 'composite_score', 'training_time']
print(valid_results[display_cols].to_string(index=False))
print('-'*110)

print('\n🏆 TOP 3 CONFIGURATIONS:')
for i, (_, row) in enumerate(valid_results.head(3).iterrows()):
    print(f"\n#{i+1}: {row['name']}")
    print(f"    {row['description']}")
    print(f"    Topics: {row['n_topics_found']} | "
          f"Outliers: {row['outlier_ratio']*100:.1f}% | "
          f"Diversity: {row['diversity']:.3f} | "
          f"U_Mass: {row['coherence_umass']:.4f}")
    print(f"    n_neighbors={row['n_neighbors']}, min_dist={row['min_dist']}, "
          f"min_cluster_size={row['min_cluster_size']}, min_samples={row['min_samples']}")
    print(f"    Composite Score: {row['composite_score']:.4f}")

best_config = valid_results.iloc[0]
print('\n' + '='*60)
print(f"🥇 BEST CONFIG: {best_config['name']}")
print('='*60)


🔬 EXPERIMENT GRID - GRANULAR TOPICS FOCUS
Total experiments: 8

-------------------------------------------------------------------------------------
Name                       n_nbr m_dist m_clust m_samp   nr_top
-------------------------------------------------------------------------------------
baseline                      15    0.0      15     10     auto
exp_1_golden_config           30   0.01       8      2     auto
exp_2_ultra_granular          30    0.0       5      1     auto
exp_3_wider_context           40    0.0       7      2     auto
exp_4_global_granular         50    0.0       8      2     auto
exp_5_spread_golden           30    0.1       8      2     auto
exp_6_force_20                30   0.01       8      2       20
exp_7_force_30                30   0.01       8      2       30
-------------------------------------------------------------------------------------
🏋️ RUNNING EXPERIMENTS
Documents: 5,000
Embeddings shape: (5000, 768)
Experiments: 8

[1/8] baseline
 

In [5]:
# ===============================================
# CELL 7: TRAIN FINAL MODEL — với post_ids thực
# ===============================================

# --- Paste nguyên Cell 7 từ notebook ---
# ===============================================
# CELL 7: TRAIN FINAL MODEL WITH BEST CONFIG
# ===============================================

print("🏆 TRAINING FINAL MODEL")
print("="*50)
print(f"Using config: {best_config['name']}")
print(f"Documents: {len(tuning_docs):,}")

# Extract best hyperparameters
best_params = {
    'n_neighbors': int(best_config['n_neighbors']),
    'min_dist': float(best_config['min_dist']),
    'min_cluster_size': int(best_config['min_cluster_size']),
    'min_samples': int(best_config['min_samples']),
    'nr_topics': best_config['nr_topics']
}

print(f"\nHyperparameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# Create final model
print("\n[1/4] Creating UMAP model...")
umap_model = UMAP(
    n_neighbors=best_params['n_neighbors'],
    n_components=5,
    min_dist=best_params['min_dist'],
    metric='cosine',
    random_state=42
)

print("[2/4] Creating HDBSCAN model...")
hdbscan_model = HDBSCAN(
    min_cluster_size=best_params['min_cluster_size'],
    min_samples=best_params['min_samples'],
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

print("[3/4] Creating BERTopic model...")
nr_topics = None if best_params['nr_topics'] == 'auto' else int(best_params['nr_topics'])
final_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=nr_topics,
    calculate_probabilities=True,
    verbose=True
)

print("[4/4] Fitting model...")
start_time = time.time()
topics, probs = final_model.fit_transform(tuning_docs, embeddings=embeddings)
training_time = time.time() - start_time

# Results
topic_info = final_model.get_topic_info()
n_topics = len([t for t in topic_info['Topic'] if t != -1])
n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].sum() if -1 in topic_info['Topic'].values else 0

print("\n" + "="*50)
print("✅ FINAL MODEL TRAINED!")
print("="*50)
print(f"Topics found: {n_topics}")
print(f"Outliers: {n_outliers} ({n_outliers/len(tuning_docs)*100:.1f}%)")
print(f"Training time: {training_time:.1f} seconds")

# Show topics
print("\n📋 TOPIC SUMMARY:")
print(topic_info[['Topic', 'Count', 'Name']].head(15).to_string(index=False))
# Sau đó THÊM phần export bên dưới:

# Tạo final_model dùng VietnameseBERTopicModel để dùng export methods
final_vm = VietnameseBERTopicModel(
    n_neighbors=int(best_config['n_neighbors']),
    n_components=5,
    min_dist=float(best_config['min_dist']),
    min_cluster_size=int(best_config['min_cluster_size']),
    min_samples=int(best_config['min_samples']),
    verbose=True
)
# Truyền embeddings đã cache → không encode lại
final_vm.topic_model = final_model   # reuse model đã train ở Cell 7
final_vm.topics_ = topics
final_vm.probs_  = probs

print("\n✅ VietnameseBERTopicModel wrapper ready for export")

2026-05-01 11:51:32,602 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


🏆 TRAINING FINAL MODEL
Using config: baseline
Documents: 5,000

Hyperparameters:
   n_neighbors: 15
   min_dist: 0.0
   min_cluster_size: 15
   min_samples: 10
   nr_topics: auto

[1/4] Creating UMAP model...
[2/4] Creating HDBSCAN model...
[3/4] Creating BERTopic model...
[4/4] Fitting model...


2026-05-01 11:51:43,888 - BERTopic - Dimensionality - Completed ✓
2026-05-01 11:51:43,889 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-01 11:51:44,243 - BERTopic - Cluster - Completed ✓
2026-05-01 11:51:44,247 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-05-01 11:51:44,646 - BERTopic - Representation - Completed ✓



✅ FINAL MODEL TRAINED!
Topics found: 2
Outliers: 0 (0.0%)
Training time: 12.1 seconds

📋 TOPIC SUMMARY:
 Topic  Count                 Name
     0   4969       0_th_nh_con_ch
     1     31 1_vs_attack_r1_ichin
🚀 GPU detected: Tesla T4

[1/4] Loading embedding model: vinai/phobert-base
✅ Embedding model loaded (device: cuda)

[2/4] Configuring UMAP
  - n_neighbors: 15
  - n_components: 5
  - min_dist: 0.0
✅ UMAP configured

[3/4] Configuring HDBSCAN
  - min_cluster_size: 15
  - min_samples: 10
✅ HDBSCAN configured

[4/4] Building BERTopic pipeline
✅ BERTopic pipeline ready

Model initialized successfully!


✅ VietnameseBERTopicModel wrapper ready for export
